# บทที่ 3 · ทำความสะอาดและสรุปข้อมูล

นักวิเคราะห์ข้อมูลใช้เวลาส่วนใหญ่ไม่ได้วาดกราฟสวย ๆ แต่ใช้กับการ *ทำความสะอาดข้อมูล* เพราะข้อมูลจริงแทบไม่เคยสมบูรณ์ — ค่าหายใน pandas แสดงเป็น `NaN` (Not a Number)

## 🗂️ การ์ดข้อมูล: titanic.csv (ข้อมูลบันทึกจริงจากประวัติศาสตร์)

| รายการ | รายละเอียด |
|---|---|
| **ที่มา** | รายชื่อผู้โดยสารเรือ RMS Titanic ที่อับปางเมื่อ 15 เมษายน ค.ศ. 1912 — ชุดข้อมูลสาธารณะเพื่อการเรียนการสอน สะท้อนบันทึกประวัติศาสตร์จริง (เผยแพร่ผ่าน Kaggle และ mirror สาธารณะ) |
| **จำนวนแถว** | 891 แถว (ผู้โดยสาร 891 คน) |
| **คอลัมน์สำคัญ** | `Survived` รอด=1 ไม่รอด=0 · `Pclass` ชั้นที่นั่ง 1/2/3 · `Name` · `Sex` · `Age` อายุ (ปี) · `SibSp` คู่สมรส+พี่น้องบนเรือ · `Parch` พ่อแม่+ลูกบนเรือ · `Fare` ค่าโดยสาร (ปอนด์) · `Cabin` ห้องโดยสาร · `Embarked` ท่าน้ำขึ้นเรือ |
| **ค่าหาย (NaN)** | มีจริงตามธรรมชาติของบันทึกเก่า: `Age` หาย 177 คน · `Cabin` หาย 687 คน · `Embarked` หาย 2 คน — เหมาะสำหรับฝึกจัดการค่าหายเป็นอย่างดี |
| **สิ่งที่ควรรู้** | 891 คนนี้เป็นส่วนหนึ่งของผู้โดยสารทั้งหมดราว 1,300+ คน · ผู้รอดชีวิตตามบันทึกประมาณ 38% ของชุดนี้ |
| **สิทธิ์การใช้** | ข้อมูลสาธารณะเพื่อการศึกษา (public domain ตามที่ Kaggle ระบุ) |

In [1]:
import pandas as pd

df = pd.read_csv("data/titanic.csv")

print(df.head())
print()
print("ขนาดตาราง:", df.shape)

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  

ข

In [2]:
# ตรวจค่าหาย: isna() แปลงเป็น True/False แล้ว sum() นับรายคอลัมน์ — combo ที่ต้องจำ!
df.info()
print()
print("จำนวนค่าหายรายคอลัมน์:")
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB

จำนวนค่าหายรายคอลัมน์:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: 

## จัดการค่า NaN — เลือก 1 จาก 3 วิธีตามบริบท

| วิธี | คำสั่ง | เหมาะกับ |
|---|---|---|
| เติมค่า | `df.fillna(ค่า)` | ค่า 0 หรือค่าที่แน่ใจ (เช่น ยอดขายวันที่ร้านปิด = 0) |
| ตัดทิ้ง | `df.dropna()` | ข้อมูลเยอะ แถวเสียนิดเดียว ตัดทิ้งไม่กระทบผลรวม |
| เติมค่ากลาง | `df["col"].fillna(df["col"].mean())` | ตัวเลขที่ไม่อยากเสียข้อมูล เช่น อายุ |

> 🧠 กับ `Age` ของเรา นักสถิตินิยมเติมด้วยค่าเฉลี่ย (วิธีที่ 3) เพราะอายุมีความหมายกับคำถาม "ใครรอดชีวิตบ้าง" และเติมค่าเฉลี่ยไม่บิดเบือนค่าเฉลี่ยรวม

In [3]:
# วิธีที่ 2: ตัดแถวที่มี NaN ทิ้งทั้งหมด — เหลือกี่แถวจาก 891?
clean = df.dropna()
print("dropna เหลือ", len(clean), "แถว จาก", len(df), "แถว (เสียไปตาม Cabin ที่หายเยอะ)")
print()

# วิธีที่ 3: เติมอายุด้วยค่าเฉลี่ย แล้วเก็บไว้ในตารางใหม่
filled = df.copy()
filled["Age"] = filled["Age"].fillna(filled["Age"].mean())
print("อายุเฉลี่ยหลังเติมค่าเฉลี่ย:", round(filled["Age"].mean(), 2), "ปี")
print("ค่าหายใน filled หลังจัดการ:", int(filled["Age"].isna().sum()))

dropna เหลือ 183 แถว จาก 891 แถว (เสียไปตาม Cabin ที่หายเยอะ)

อายุเฉลี่ยหลังเติมค่าเฉลี่ย: 29.7 ปี
ค่าหายใน filled หลังจัดการ: 0


## เรียงลำดับ และนับความถี่

In [4]:
# sort_values: ใครจ่ายค่าโดยสารแพงที่สุด
print(df.sort_values("Fare", ascending=False)[["Name", "Fare"]].head(3))
print()

# value_counts: นับความถี่ของหมวดหมู่
print(df["Sex"].value_counts())
print()
print(df["Pclass"].value_counts().sort_index())

                                   Name      Fare
679  Cardeza, Mr. Thomas Drake Martinez  512.3292
258                    Ward, Miss. Anna  512.3292
737              Lesurer, Mr. Gustave J  512.3292

Sex
male      577
female    314
Name: count, dtype: int64

Pclass
1    216
2    184
3    491
Name: count, dtype: int64


## `groupby` — แยกกลุ่ม แล้วสรุป (เหมือน Pivot Table ของ Excel)

แนวคิด 3 ขั้น: **แยก**ข้อมูลตามคอลัมน์ที่ต้องการ → **คำนวณ** (sum, mean, count…) → **รวม**ผลเป็นตาราง

คำถามประวัติศาสตร์ที่ทุกคนอยากรู้: *บนเรือลำนั้น เพศและชั้นที่นั่งมีผลกับโอกาสรอดชีวิตจริงหรือ?*

In [5]:
# อัตราการรอดชีวิตแยกตามเพศ (Survived เป็น 0/1 -> mean() คือ % ที่รอด)
print("อัตรารอดชีวิตตามเพศ:")
print((df.groupby("Sex")["Survived"].mean() * 100).round(1), "%")
print()

# อัตราการรอดชีวิตแยกตามชั้นที่นั่ง
print("อัตรารอดชีวิตตามชั้นที่นั่ง:")
print((df.groupby("Pclass")["Survived"].mean() * 100).round(1), "%")

อัตรารอดชีวิตตามเพศ:
Sex
female    74.2
male      18.9
Name: Survived, dtype: float64 %

อัตรารอดชีวิตตามชั้นที่นั่ง:
Pclass
1    63.0
2    47.3
3    24.2
Name: Survived, dtype: float64 %


พบข้อมูลที่น่าตกใจ: **ผู้หญิงรอดชีวิตราว 3 เท่าของผู้ชาย** และคนชั้น 1 มีโอกาสรอดมากกว่าชั้น 3 อย่างชัดเจน — นี่คือพลังของ groupby: ทำให้เห็น "ความไม่เท่าเทียม" จากตัวเลขจริงใน 2 บรรทัด

## 🎯 ภารกิจบทที่ 3 (ส่งคำตอบใน Moodle)

**มีผู้หญิงที่รอดชีวิตทั้งหมดกี่คน** ในชุดข้อมูลนี้? (ตอบเป็นจำนวนเต็ม)

*ใบ้: กรอง 2 เงื่อนไข `(df["Sex"] == "female") & (df["Survived"] == 1)` แล้วนับแถว*

In [6]:
# ภารกิจ 3: จำนวนผู้หญิงที่รอดชีวิต
women_survived = df[(df["Sex"] == "female") & (df["Survived"] == 1)]
print("ผู้หญิงที่รอดชีวิต =", len(women_survived), "คน")
print("คำตอบภารกิจบทที่ 3 =", len(women_survived))

ผู้หญิงที่รอดชีวิต = 233 คน
คำตอบภารกิจบทที่ 3 = 233
